In [ ]:
from pathlib import Path
import importlib
import sys

# Resolve project root robustly whether notebook runs from project root or notebooks/
PROJECT_ROOT = (
    Path.cwd() if (Path.cwd() / "config" / "config.yaml").exists() else Path.cwd().parent
).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_module = importlib.import_module("data.load")
preprocess_module = importlib.import_module("data.preprocess")
health_index_module = importlib.import_module("features.health_index")
velocity_module = importlib.import_module("features.velocity")
variability_module = importlib.import_module("features.variability")
clustering_module = importlib.import_module("model.clustering")

load_dataset = load_module.load_dataset
load_config = load_module.load_config
preprocess_train = preprocess_module.preprocess_train
preprocess_test = preprocess_module.preprocess_test
build_health_index = health_index_module.build_health_index
build_velocity = velocity_module.build_velocity
build_variability = variability_module.build_variability
build_clustering = clustering_module.build_clustering

config = load_config(PROJECT_ROOT / "config" / "config.yaml")
config["dataset"]["raw_path"] = str(PROJECT_ROOT / "data" / "raw")

train_raw, test_raw, _ = load_dataset(config)
train_proc, scaler, sensor_cols = preprocess_train(train_raw, config)
test_proc = preprocess_test(test_raw, config, scaler)
train_hi, test_hi, hi_art = build_health_index(train_proc, test_proc, config)
train_vel, test_vel, velocity_artifacts = build_velocity(train_hi, test_hi, config)
train_var, test_var, var_art = build_variability(train_vel, test_vel, config)
train_cl, test_cl, cl_art = build_clustering(train_var, test_var, config)

# Shape checks
assert train_cl.shape[0] == 20631
assert test_cl.shape[0] == 13096
assert "risk_state" in train_cl.columns

# All three labels must appear in training data
assert set(train_cl["risk_state"].unique()) == {"Healthy", "Degrading", "Critical"}

# Cluster ordering — Healthy should have highest mean HI
mean_hi = train_cl.groupby("risk_state", observed=False)["health_index"].mean()
assert mean_hi["Healthy"] > mean_hi["Degrading"] > mean_hi["Critical"], \
    f"Cluster ordering violated: {mean_hi.to_dict()}"

# Silhouette score must be reasonable
assert cl_art.silhouette > 0.3, f"Silhouette too low: {cl_art.silhouette:.3f}"

# No NaN in risk_state
assert train_cl["risk_state"].isna().sum() == 0

print("✅ All checks passed")
print(f"   Silhouette score : {cl_art.silhouette:.4f}")
print("\nCluster distribution (train):")
print(train_cl["risk_state"].value_counts())
print("\nCentroid summary (original feature space):")
print(cl_art.centroid_summary.to_string(index=False))

[load] Train shape : (20631, 26)
[load] Test shape  : (13096, 26)
[load] RUL entries : 100
✅ All checks passed
   Silhouette score : 0.4005

Cluster distribution (train):
risk_state
Healthy      12595
Degrading     6105
Critical      1931
Name: count, dtype: int64

Centroid summary (original feature space):
risk_state  health_index  HI_velocity  HI_variability
   Healthy      0.722986    -0.000675        0.210129
 Degrading      0.514162    -0.003546        0.272143
  Critical      0.310073    -0.008325        0.426241
